# Who Wants to Be a Millionaire — llama-cpp-python backend

This notebook runs the competition client on Google Colab using **llama-cpp-python**  
(no Ollama required — the model runs directly inside the Python process via GGUF).

**Runtime**: set to **GPU → T4** (or better) before running.  
Runtime → Change runtime type → Hardware accelerator → GPU

## 1 — Environment setup

Installs llama-cpp-python with CUDA support, then all other dependencies.  
This cell takes ~3-5 minutes on first run.

In [1]:
import subprocess, sys

# Check whether a GPU is visible
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NOT FOUND — set Runtime to GPU!')

# Pre-built CUDA wheel (no compilation needed)
print('\nInstalling llama-cpp-python (pre-built CUDA wheel)...')
!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    --quiet

# Remaining Python dependencies
print('Installing remaining dependencies...')
!pip install faster-whisper huggingface-hub requests ddgs \
             sentence-transformers trafilatura numpy scipy psutil python-dotenv --quiet

print('\nAll packages installed.')

GPU: Tesla T4, 15360 MiB

Installing llama-cpp-python (pre-built CUDA wheel)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 635.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
Installing remaining dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━

## 2 — Mount Google Drive (optional but recommended)

Caches the GGUF model file on Drive so you don't re-download it every session.  
Skip this cell if you prefer to download fresh each time.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODEL_DIR = '/content/drive/MyDrive/llm_models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print('Model cache directory:', DRIVE_MODEL_DIR)

Mounted at /content/drive
Model cache directory: /content/drive/MyDrive/llm_models


## 3 — Download GGUF model

We use `huggingface_hub.hf_hub_download` which respects a local cache dir.  
Change `REPO_ID` and `FILENAME` to any GGUF you want — examples:

| Model | repo_id | filename |
|---|---|---|
| Qwen2.5-7B-Q4 | `Qwen/Qwen2.5-7B-Instruct-GGUF` | `qwen2.5-7b-instruct-q4_k_m.gguf` |
| Gemma-3-4B-Q4 | `google/gemma-3-4b-it-GGUF` | `gemma-3-4b-it-q4_k_m.gguf` |
| Llama-3.1-8B-Q4 | `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF` | `Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf` |

A Q4_K_M 7-8B model fits comfortably in T4's 15 GB VRAM.

In [3]:
from huggingface_hub import hf_hub_download
import os

# ── CONFIGURE THIS ──────────────────────────────────────────────────────────
REPO_ID  = 'bartowski/Qwen2.5-7B-Instruct-GGUF'
FILENAME = 'Qwen2.5-7B-Instruct-Q4_K_M.gguf'
MODEL_NAME = 'qwen2.5'
# ────────────────────────────────────────────────────────────────────────────

# Use Drive cache if mounted, otherwise Colab's local /root/.cache
DRIVE_MODEL_DIR = '/content/drive/MyDrive/llm_models'
cache_dir = DRIVE_MODEL_DIR if os.path.isdir(DRIVE_MODEL_DIR) else None

print(f'Downloading {FILENAME} from {REPO_ID}...')
print(f'Cache dir: {cache_dir or "~/.cache/huggingface"}')

MODEL_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    cache_dir=cache_dir,
)
print(f'\nModel ready at: {MODEL_PATH}')
print(f'Size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB')

Cache dir: /content/drive/MyDrive/llm_models


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Qwen2.5-7B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]


Model ready at: /content/drive/MyDrive/llm_models/models--bartowski--Qwen2.5-7B-Instruct-GGUF/snapshots/8911e8a47f92bac19d6f5c64a2e2095bd2f7d031/Qwen2.5-7B-Instruct-Q4_K_M.gguf
Size: 4.68 GB


## 4 — Clone / upload the client code

The easiest way is to clone your fork of the repo.  
Replace the URL with your own if needed, or upload the `colab-llama/` folder manually.

In [5]:
import os, sys

CODE_DIR = '/content/drive/MyDrive/final notebook polimillionaire'

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.chdir(CODE_DIR)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

Working directory: /content/drive/MyDrive/final notebook polimillionaire
Files: ['wrong_answers.py', 'rag.py', 'requirements.txt', 'ollama_client.py', 'wrong_answers.jsonl', 'speech_client.py', 'millionaire_client', '__pycache__', 'run-comp.ipynb']


## 5 — Configure environment

Set credentials and model settings.  
Use Colab Secrets (🔑 icon) to store `USERNAME` and `PASSWORD` instead of hardcoding.

In [6]:
import os

# ── API ──────────────────────────────────────────────────────────────────────
API_URL  = 'http://131.175.15.22:51111/'

# Load from Colab Secrets if available, otherwise fill in manually
try:
    from google.colab import userdata
    USERNAME = userdata.get('USERNAME')
    PASSWORD = userdata.get('PASSWORD')
    print('Credentials loaded from Colab Secrets.')
except Exception:
    USERNAME = 'suf'   # ← change if not using Secrets
    PASSWORD = 'NLP2026!'   # ← change if not using Secrets

# ── Model ────────────────────────────────────────────────────────────────────
# MODEL_PATH / MODEL_NAME were set in cell 3; set them as env vars so
# ollama_client.py picks them up via os.getenv()
os.environ['MODEL_PATH']    = MODEL_PATH
os.environ['MODEL_NAME']    = MODEL_NAME
os.environ['N_CTX']         = '8192'   # context window
os.environ['N_GPU_LAYERS']  = '-1'     # -1 = offload all layers to GPU
os.environ['N_THREADS']     = '4'      # CPU threads for non-GPU ops

# ── RAG ──────────────────────────────────────────────────────────────────────
os.environ['ENABLE_RAG'] = 'true'   # set to 'false' to disable RAG

print(f'API:   {API_URL}')
print(f'User:  {USERNAME}')
print(f'Model: {MODEL_NAME}  ({MODEL_PATH})')
print(f'n_ctx={os.environ["N_CTX"]}  n_gpu_layers={os.environ["N_GPU_LAYERS"]}')

API:   http://131.175.15.22:51111/
User:  suf
Model: qwen2.5  (/content/drive/MyDrive/llm_models/models--bartowski--Qwen2.5-7B-Instruct-GGUF/snapshots/8911e8a47f92bac19d6f5c64a2e2095bd2f7d031/Qwen2.5-7B-Instruct-Q4_K_M.gguf)
n_ctx=8192  n_gpu_layers=-1


## 6 — Login

In [7]:
from millionaire_client import MillionaireClient, AuthenticationError

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as {user.username!r}  (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as 'suf'  (role: student)


## 7 — Quick model test

Verify the model loads and answers correctly before running a full game.  
First call is slow (cold load); subsequent calls reuse the cached instance.

In [8]:
from ollama_client import ask_gemma

question = (
    'A researcher plans a study to examine long-term confidence in the U.S. economy among '
    'the adult population. She obtains a simple random sample of 30 adults as they leave a '
    'Wall Street office building one weekday afternoon. All but two of the adults agree to '
    'participate in the survey. Which of the following conclusions is correct?'
)
options = {
    0: 'Selection bias makes this a poorly designed survey.',
    1: 'The high response rate makes this a well-designed survey.',
    2: 'A voluntary response study like this gives too much emphasis to persons with strong opinions.',
    3: 'Proper use of chance as evidenced by the simple random sample makes this a well-designed survey.',
}

answer, reasoning, resources = ask_gemma(question, options)
print(f'Answer option: {answer}')
print(f'Throughput   : {resources["tps"]:.1f} tok/s')
print(f'Time         : {resources["elapsed"]:.2f}s')
print(f'\nReasoning:\n{reasoning[:500]}')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[RAG] Source : https://www.numerade.com/ask/question/a-researcher-plans-a-study-to-examine-long-term-confidence-in-the-us-economy-among-the-adult-population-she-obtains-a-simple-random-sample-of-30-adults-as-they-leave-a-wall-street-office-building-one-/
[RAG] Score  : 8.01
[RAG] Chunks : 2
[RAG] Context: A researcher plans a study to examine long-term confidence in the U.S. economy among the adult population. She obtains a simple random sample of 30 adults as they leave a Wall Street office building one weekday afternoon.

VIDEO ANSWER: He reaches her plans to study to examine long -term confidence ...



llama_context: n_ctx_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



[Answer]
Let's analyze the options step by step:

1. **Option 0: Selection bias makes this a poorly designed survey.**
   - The sample is a simple random sample of 30 adults leaving a Wall Street office building. This method of sampling is generally considered a good practice for reducing selection bias.
   - The sample size of 30 is relatively small, but it is not necessarily indicative of selection bias without further context about the population and the study's requirements.
   - The fact that all but two adults agreed to participate does not necessarily indicate selection bias, as it could be due to the nature of the sample (e.g., people leaving a financial office might be more likely to agree).

2. **Option 1: The high response rate makes this a well-designed survey.**
   - The response rate is 93.3% (28 out of 30), which is indeed high. However, a high response rate does not necessarily make the survey well-designed. The sample is still a simple random sample, and the key issue

## 8 — Play a single text game

Competition IDs: **0** = Entertainment · **1** = History & Politics · **2** = Science & Nature · **3** = Mathematics

In [ ]:
from ollama_client import ask_gemma
from rag import fetch_rag_context, should_use_rag
from wrong_answers import save_wrong_answer


def play_game(game, use_rag=True, username='unknown', competition_id=None):
    """Play one game session autonomously (text mode)."""
    all_resources = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print('No question available. Game may have ended.')
            break

        print(f'\n--- Level {game.current_level} ---')
        print(f'Q: {question.text}')

        question_options = {int(opt.id): opt.text for opt in question.options}
        for oid, otxt in question_options.items():
            print(f'  {oid}: {otxt}')

        time_left = game.time_remaining
        if time_left:
            print(f'Time remaining: {time_left:.1f}s')

        # RAG
        context = None
        if use_rag and should_use_rag(question.text):
            context = fetch_rag_context(question.text, question_options, k=5)

        answer_input, reasoning, resources = ask_gemma(question.text, question_options, context=context)
        answer_id = int(answer_input)
        all_resources.append(resources)

        result = game.answer(answer_id)

        if result.correct:
            print(f'✓ CORRECT!  Earned: ${result.earned_amount:,.2f}')
            if result.game_over:
                print('🎉 CONGRATULATIONS! Game complete!')
        elif result.timed_out:
            print('⏰ TIMED OUT!')
            save_wrong_answer(
                username=username, competition_id=competition_id,
                level=game.current_level, question=question.text,
                options=question_options, reasoning=reasoning,
                answer_given=answer_input, rag_context=context,
                resources=resources, timed_out=True,
            )
        else:
            print(f'✗ WRONG!  Earnings locked at ${result.earned_amount:,.2f}')
            save_wrong_answer(
                username=username, competition_id=competition_id,
                level=game.current_level, question=question.text,
                options=question_options, reasoning=reasoning,
                answer_given=answer_input, rag_context=context,
                resources=resources, timed_out=False,
            )

    # Summary
    print(f'\n=== Game Summary ===')
    print(f'Reached level : {game.current_level}')
    print(f'Total earnings: ${game.earned_amount:,.2f}')
    if all_resources:
        n = len(all_resources)
        print(f'Avg time      : {sum(r["elapsed"] for r in all_resources)/n:.2f}s')
        print(f'Avg throughput: {sum(r["tps"] for r in all_resources)/n:.1f} tok/s')


# ── Run ──────────────────────────────────────────────────────────────────────
COMP_ID = 0   # 0=Entertainment, 1=History, 2=Science, 3=Math
USE_RAG = True

game = client.game.start(competition_id=COMP_ID)
play_game(game, use_rag=USE_RAG, username=USERNAME, competition_id=COMP_ID)


--- Level 1 ---
Q: What term describes the monster in The Babadook?
  0: Ghost
  1: Babadook
  2: Werewolf
  3: Vampire
Time remaining: 29.8s


[RAG] Source : https://en.wikipedia.org/wiki/The_Babadook
[RAG] Score  : 9.74
[RAG] Chunks : 2
[RAG] Context: The Babadook
The Babadook is a 2014 Australian psychological horror film written and directed by Jennifer Kent in her feature directorial debut, based on her 2005 short film Monster. Starring Essie Davis, Noah Wiseman, Daniel Henshall, Hayley McElhinney, Barbara West, and Ben Winspear, the film foll...


[Answer]
The term that describes the monster in The Babadook is "Babadook."

Therefore my answer is option 1.

✓ CORRECT!  Earned: $100.00

--- Level 2 ---
Q: Which of the following best describes Jay-Z's role in the music industry as of 2017?
  0: CEO of Roc Nation
  1: President of Def Jam Recordings
  2: President of Reebok
  3: President of Universal Music Group
Time remaining: 29.2s


[RAG] Source : https://www.strikemagazines.com/blog-2-1/the-billion-dollar-model-how-jay-z-gives-a-blueprint-for-artists-in-the-modern-music-industry
[RAG] Score  : 6.16
[RAG] Chunks : 2
[RAG] Context: At one point, Jay-Z retired from rapping, marking the occasion with a private event and releasing “The Black Album.” He then took over as president of Def Jam. This illustrates the importance of having a timeline and recognizing that an artist’s journey is more than just the creative work they love ...


[Answer]
Let's analyze the options step by step:

1. **Option 0: CEO of Roc Nation**
   - The retrieved context does not mention Jay-Z being the CEO of Roc Nation. It only mentions his role as president of Def Jam.

2. **Option 1: President of Def Jam Recordings**
   - The retrieved context clearly states that Jay-Z took over as president of Def Jam. This is a fact that is supported by the context.

3. **Option 2: President of Reebok**
   - The retrieved context does



⏰ TIMED OUT!

=== Game Summary ===
Reached level : 2
Total earnings: $100.00
Avg time      : 12.01s
Avg throughput: 9.7 tok/s


## 9 — Multi-run text experiment

Plays `RUNS` games in a row for the chosen competition and prints a summary table.

In [ ]:
import time, io, contextlib, sys

COMP_ID = 0    # 0=Entertainment, 1=History, 2=Science, 3=Math
RUNS    = 5
PAUSE   = 10   # seconds between games
USE_RAG = True

results = []   # (level_reached, earned)

for run in range(1, RUNS + 1):
    sys.__stdout__.write(f'\n[{run}/{RUNS}] Starting game...\n')
    sys.__stdout__.flush()

    game = client.game.start(competition_id=COMP_ID)

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        play_game(game, use_rag=USE_RAG, username=USERNAME, competition_id=COMP_ID)

    level   = game.current_level
    earned  = game.earned_amount
    results.append((level, earned))
    sys.__stdout__.write(f'  → Level {level}  ${earned:,.2f}\n')
    sys.__stdout__.flush()

    if run < RUNS:
        time.sleep(PAUSE)

# Summary
print(f"\n{'='*42}")
print(f"  SUMMARY  {RUNS} games  comp={COMP_ID}")
print(f"{'='*42}")
for i, (lv, ea) in enumerate(results, 1):
    print(f"  Game {i:>2}: level {lv:>2}   ${ea:>12,.2f}")
print(f"  {'─'*36}")
levels   = [r[0] for r in results]
earnings = [r[1] for r in results]
print(f"  Avg level  : {sum(levels)/len(levels):.1f}")
print(f"  Avg earned : ${sum(earnings)/len(earnings):>12,.2f}")
print(f"  Best       : ${max(earnings):>12,.2f}")
print(f"  Worst      : ${min(earnings):>12,.2f}")

## 10 — Single speech game

The server streams audio for each question/option.  
`faster-whisper` transcribes them locally before passing to the model.  
Whisper model downloads automatically on first call (~39 MB for `tiny`, ~142 MB for `base`).

In [ ]:
from speech_client import play_game_speech

COMP_ID = 0   # 0=Entertainment, 1=History, 2=Science, 3=Math

game = client.game.start(competition_id=COMP_ID, mode='speech')
play_game_speech(game, username=USERNAME, competition_id=COMP_ID, use_rag=True)

## 11 — Multi-run speech experiment

Same as cell 9 but uses speech mode.  
Console output is suppressed per game to keep the log clean.

In [ ]:
import time, io, contextlib, logging, sys
import speech_client
from speech_client import play_game_speech

COMP_ID = 0    # 0=Entertainment, 1=History, 2=Science, 3=Math
RUNS    = 5
PAUSE   = 15   # seconds between games

results = []
_orig_ipython = speech_client._IPYTHON_AVAILABLE

for run in range(1, RUNS + 1):
    sys.__stdout__.write(f'\n[{run}/{RUNS}] Starting speech game...\n')
    sys.__stdout__.flush()

    game = client.game.start(competition_id=COMP_ID, mode='speech')

    speech_client._IPYTHON_AVAILABLE = False   # suppress audio widgets in Colab
    logging.disable(logging.CRITICAL)
    with contextlib.redirect_stdout(io.StringIO()):
        play_game_speech(game, username=USERNAME, competition_id=COMP_ID, use_rag=True)
    logging.disable(logging.NOTSET)
    speech_client._IPYTHON_AVAILABLE = _orig_ipython

    level  = game.current_level
    earned = game.earned_amount
    results.append((level, earned))
    sys.__stdout__.write(f'  → Level {level}  ${earned:,.2f}\n')
    sys.__stdout__.flush()

    if run < RUNS:
        sys.__stdout__.write(f'  Pausing {PAUSE}s...\n')
        sys.__stdout__.flush()
        time.sleep(PAUSE)

# Summary
print(f"\n{'='*42}")
print(f"  SPEECH SUMMARY  {RUNS} games  comp={COMP_ID}")
print(f"{'='*42}")
for i, (lv, ea) in enumerate(results, 1):
    print(f"  Game {i:>2}: level {lv:>2}   ${ea:>12,.2f}")
print(f"  {'─'*36}")
levels   = [r[0] for r in results]
earnings = [r[1] for r in results]
print(f"  Avg level  : {sum(levels)/len(levels):.1f}")
print(f"  Avg earned : ${sum(earnings)/len(earnings):>12,.2f}")
print(f"  Best       : ${max(earnings):>12,.2f}")
print(f"  Worst      : ${min(earnings):>12,.2f}")

## 12 — Leaderboard

In [ ]:
COMP_ID = 0

lb = client.leaderboard.get(COMP_ID)
print(f'Leaderboard: {lb.competition.name}\n')
print(f'{"Rank":>4}  {"Username":<20}  {"Level":>5}  {"Score":>12}')
print('─' * 48)
for rank, entry in enumerate(lb.entries[:20], 1):
    marker = ' ◄' if entry.username == USERNAME else ''
    print(f'{rank:>4}  {entry.username:<20}  {entry.reached_level:>5}  {entry.score:>12,.2f}{marker}')